# Agentic AI exercise notebook

This notebook is a guided lab for building a small course-policy assistant one capability at a time.
We keep the task fixed and change the system around the model so each stage has a clear purpose.

## Learning goals
- measure what a plain LLM can and cannot do on course-specific questions
- see why retrieval helps policy lookup and cross-document synthesis
- see why deterministic tools are better than free-form generation for grade calculations
- see why an evidence check improves behavior on unsupported questions

## How to use this notebook
At each stage:
1. look at the benchmark snapshot
2. inspect one representative success and the remaining failures
3. explain which new capability caused the change

In [ ]:
#@title Install dependencies and load the lab { display-mode: "form" }
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    REPO_URL = "https://github.com/MAS-AID-AI-Project/weekend4_agentic_public.git"

    if os.path.exists("weekend4_agentic_public"):
        !cd weekend4_agentic_public && git pull
    else:
        !git clone {REPO_URL}
    os.chdir("weekend4_agentic_public/exercises/WE4_2_course_grader")

    # Load the OpenRouter API key into the environment
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

!pip install -q pandas python-dotenv

import agentic_ai_progressive_support_openrouter as lab

lab.bootstrap_notebook()

## Roadmap

We start by reading the course corpus and the benchmark so the evaluation target is concrete.
Then we run the baseline and add capabilities only when a failure mode justifies them.

- **Stage 0:** LLM only
- **Stage 1:** add retrieval
- **Stage 2:** let the model choose tools
- **Stage 3:** add an evidence check

The benchmark is intentionally small so you can inspect answers directly instead of relying only on one aggregate score.

## 1. Inspect the tiny local corpus

The support script loads a small set of markdown files from `data_agentic_course/`.
The documents overlap on purpose: some policies are repeated across files, some questions require combining files, and some answers are missing entirely.

That overlap is what makes retrieval, tool use, and abstention behavior easy to observe later.

In [2]:
lab.show_corpus()

### `faq.md`

FAQ:
Office hours are held on Tuesdays 14:00-16:00 and Thursdays 10:00-11:00.

Collaboration:
Students may discuss homework concepts, but all submitted work must be written individually.
For the project, code may be shared only within the approved team.

Extensions:
Documented medical or family emergencies should be discussed with the instructor as early as possible.

Attendance:
Lecture attendance is recommended but not required.


---

### `grading.md`

Homework policy:
There are 5 homework assignments. The lowest homework score is dropped before computing
the homework average.

Late policy summary:
Each student receives 3 late days for the semester. A maximum of 2 late days may be used
on any single homework. Late days cannot be used for the final project.

Regrade requests:
A regrade request must be submitted within 7 calendar days of the score release.

Project grading:
The final project is worth 15% of the course grade.
Project milestones are graded separately but roll into the final project score.


---

### `late_policy.md`

Detailed late policy:
Late days apply only to homework assignments.
No late days may be used for the midterm, final exam, or final project.

Penalty after late days:
After all applicable late days are exhausted, the penalty is 10 percentage points per day,
up to 3 additional days. After that, the submission receives zero credit.

Examples:
- 1 day late with 1 late day remaining: no penalty.
- 3 days late with 2 late days available for that homework: 10-point penalty.
- 5 days late with no late days available: zero credit.


...

---

### `project.md`

Project rules:
Students may work individually or in teams of 2.
Teams of 3 are not allowed.

Programming languages:
Python is fully supported.
Julia is permitted for the final project only if the team also submits a short README
explaining how to run the code.
R is permitted under the same condition.

Submission:
The final project submission consists of a report, code, and a 3-minute demo video.


---

### `syllabus.md`

Course: Introduction to Agentic AI

Assessment overview:
- Homeworks: 40%
- Midterm: 20%
- Final exam: 25%
- Project: 15%

Letter grade thresholds:
- A: 93 and above
- A-: 90 to 92.99
- B+: 87 to 89.99
- B: 83 to 86.99
- B-: 80 to 82.99

Communication:
Students should use the course forum for general questions.
Private grading issues should be sent by email to the teaching staff.


---

## 2. Inspect the benchmark before building anything

The benchmark has four question types:
- `factual_in_doc`
- `cross_doc`
- `numeric_requires_tool`
- `unanswerable`

Reading the questions first matters pedagogically.
You should know what counts as success before you interpret any stage output.

In [3]:
benchmark_df = lab.show_benchmark()

### Benchmark questions

id,type,question,expected_answer
1,factual_in_doc,How many late days does each student receive for the semester?,Each student receives 3 late days for the semester.
2,factual_in_doc,What is the maximum number of late days allowed on a single homework?,A maximum of 2 late days may be used on a single homework.
3,factual_in_doc,Within how many calendar days must a regrade request be submitted?,A regrade request must be submitted within 7 calendar days of score release.
4,factual_in_doc,What is the weight of the final project in the course grade?,The final project is worth 15% of the course grade.
5,cross_doc,"Can teams of 3 submit the final project, and are late days allowed for that project?","No. Teams of 3 are not allowed, and late days cannot be used for the final project."
6,factual_in_doc,"Where are office hours normally held, and what changes during exam week?","Office hours are normally held in Room 3.14, and during exam week they move online by video call."
7,cross_doc,"Is lecture attendance required, and where should private grading issues be sent?","Lecture attendance is recommended but not required, and private grading issues should be sent by email to the teaching staff."
8,factual_in_doc,"Is Julia allowed for the project, and under what condition?",Yes. Julia is allowed for the final project if the team also submits a short README explaining how to run the code.
9,numeric_requires_tool,"My homework scores are 92.6, 87.1, 78.4, 90.3, and 94.8. I still have 3 late days left for the semester. Homework 1 was 2 days late, Homework 3 was 1 day late, and Homework 5 was 4 days late. If the remaining late days are used optimally under the course policy and the lowest homework is dropped, what homework average counts?","After allocating the remaining late days optimally and dropping the lowest homework, the counted homework average is 87.10."
10,numeric_requires_tool,"My homework scores are 88.7, 91.4, 84.2, 79.9, and 95.1. I still have 3 late days left for the semester. Homework 2 was 1 day late, Homework 4 was 3 days late, and Homework 5 was 2 days late. My midterm is 86.35 and my project is 91.8. If the remaining late days are used optimally under the course policy, what exact final exam score do I need for an A- (90 overall)?","After allocating the remaining late days optimally, you need 92.08 on the final exam to reach 90.00 overall."


## 3. Stage 0: LLM only

In the baseline, the model gets only the user question and a plain assistant prompt.
It does not see the local corpus, it does not get calculator output, and it does not pass through any notebook-side support check.

What to expect:
- some factual answers may be plausible but unsupported
- cross-document questions should be especially weak
- numeric grade answers may be inconsistent or wrong
- unanswerable questions may trigger guessing instead of abstention

In [4]:
lab.show_stage_pseudocode("Stage 0 - LLM only", lab.AgentConfig())

### Stage 0 - LLM only pseudocode

```python
cfg = AgentConfig(
    enable_retrieve=False,
    enable_tool=False,
    enable_evidence_check=False,
)

for item in benchmark:
    evidence = []
    tool_result = None
    answer = None

    while True:
        allowed_tools = ["finish"]
        # allowed_tools.append('retrieve')
        # allowed_tools.append('grade_calculator')

        if allowed_tools == ["finish"]:
            # Stage 0: there is nothing to choose, so use a direct LLM call.
            answer = llm_text(question_only_prompt)
            break

        decision = planner_llm(item, evidence, tool_result, allowed_tools)

        if decision.action == "retrieve":
            evidence = retrieve(decision.query)
            continue
        if decision.action == "grade_calculator":
            tool_result = grade_calculator(...)
            continue
        if decision.action == "finish":
            answer = render_final_answer(evidence, tool_result)
            break

    if cfg.enable_evidence_check:
        answer = enforce_evidence_gate(answer, evidence, tool_result)
    else:
        # no evidence gate in this stage
        pass

    score_answer(item, answer)
```

In [5]:
stage_0_cfg = lab.AgentConfig()
stage_0_results = lab.run_stage_analysis("Stage 0 - LLM only", stage_0_cfg)

### Benchmark snapshot

### Benchmark accuracy

Category,Correct,Total,Accuracy
Factual,1,7,14.3%
Cross-doc,0,2,0.0%
Numeric,0,4,0.0%
Abstention,1,3,33.3%
Overall,2,16,12.5%


### Representative correct answer

### Failure cases

### Stage 0 takeaway

This is the reference point for the rest of the notebook.
Any later improvement should be explained by a capability we explicitly add, not by vague "agent magic."

## 4. Retrieval: what changes in Stage 1

Before we run Stage 1, look at the retriever in action on the first cross-document benchmark question.
The retriever is intentionally simple so you can reason about it:

- split each document into overlapping character chunks
- tokenize the question and each chunk with a lexical tokenizer
- score by keyword overlap plus a small repetition bonus
- return up to `k=3` chunks, preferring document diversity first

This is not an embeddings demo.
The point is to show that even a readable lexical retriever can materially improve grounded answers.

In [6]:
retrieval_preview_question = lab.BENCHMARK[4]["question"]
_ = lab.show_retrieval(retrieval_preview_question, k=3)

#### Hit 1: `grading.md`

Homework policy:
There are 5 homework assignments. The lowest homework score is dropped before computing
the homework average.

Late policy summary:
Each student receives 3 late days for the semester. A maximum of 2 late days may be used
on any single homework. Late days cannot be used for the final project.

Regrade requests:
A regrade request must be submitted within 7 calendar days of the score release.

Project grading:
The final project is w

---

#### Hit 2: `project.md`

Project rules:
Students may work individually or in teams of 2.
Teams of 3 are not allowed.

Programming languages:
Python is fully supported.
Julia is permitted for the final project only if the team also submits a short README
explaining how to run the code.
R is permitted under the same condition.

Submission:
The final project submission consists of a report, code, and a 3-minute demo video.

---

#### Hit 3: `late_policy.md`

Detailed late policy:
Late days apply only to homework assignments.
No late days may be used for the midterm, final exam, or final project.

Penalty after late days:
After all applicable late days are exhausted, the penalty is 10 percentage points per day,
up to 3 additional days. After that, the submission receives zero credit.

Examples:
- 1 day late with 1 late day remaining: no penalty.
- 3 days late with 2 late days available for that homewo

---

## 5. Stage 1: Add retrieval

Now the notebook inserts the retrieved chunks into the prompt before the model answers.
The model still has no calculator and no evidence gate.

What to watch for:
- factual questions should improve first
- cross-document questions should improve when the right chunks are retrieved
- unsupported answers can still slip through because retrieval is not the same thing as verification

In [7]:
lab.show_stage_pseudocode("Stage 1 - Add retrieval", lab.AgentConfig(enable_retrieve=True))

### Stage 1 - Add retrieval pseudocode

```python
cfg = AgentConfig(
    enable_retrieve=True,
    enable_tool=False,
    enable_evidence_check=False,
)

for item in benchmark:
    evidence = []
    tool_result = None
    answer = None

    while True:
        allowed_tools = ["finish"]
        allowed_tools.append('retrieve')
        # allowed_tools.append('grade_calculator')

        if allowed_tools == ["finish"]:
            # Stage 0: there is nothing to choose, so use a direct LLM call.
            answer = llm_text(question_only_prompt)
            break

        decision = planner_llm(item, evidence, tool_result, allowed_tools)

        if decision.action == "retrieve":
            evidence = retrieve(decision.query)
            continue
        if decision.action == "grade_calculator":
            tool_result = grade_calculator(...)
            continue
        if decision.action == "finish":
            answer = render_final_answer(evidence, tool_result)
            break

    if cfg.enable_evidence_check:
        answer = enforce_evidence_gate(answer, evidence, tool_result)
    else:
        # no evidence gate in this stage
        pass

    score_answer(item, answer)
```

In [8]:
stage_1_cfg = lab.AgentConfig(enable_retrieve=True)
stage_1_results = lab.run_stage_analysis("Stage 1 - Add retrieval", stage_1_cfg)

### Benchmark snapshot

### Benchmark accuracy

Category,Correct,Total,Accuracy
Factual,7,7,100.0%
Cross-doc,2,2,100.0%
Numeric,0,4,0.0%
Abstention,1,3,33.3%
Overall,10,16,62.5%


### Representative correct answer

### Failure cases

### Stage 1 takeaway

Retrieval usually improves grounding, but it does not guarantee correctness.
The model still has to synthesize the evidence, and it can still overgeneralize from partial support.

## 6. Tool use: what changes in Stage 2

Stage 2 is the first genuinely notebook-side agentic setup.
The notebook still owns the tools, but the model chooses the next action from a small menu: `retrieve`, `grade_calculator`, or `finish`.

A simplified loop is:

```python
while True:
    decision = planner_llm(question, evidence, tool_result, allowed_tools)
    if decision == "retrieve":
        evidence = retrieve(...)
    elif decision == "grade_calculator":
        tool_result = grade_calculator(...)
    else:
        answer = render_final_answer(...)
        break
```

The key teaching point is not autonomy for its own sake.
It is that the model can now decide when free-form text is inappropriate and when a deterministic calculation is better.

## 7. Stage 2: Let the model choose tools

Retrieval helps with policy lookup, but numeric questions still need exact computation.
In this stage, the model can choose when to retrieve, when to call the calculator, and when to finish.

What to watch for:
- numeric questions should improve most
- the workflow changes from notebook-decided to model-decided
- the notebook still executes the tools, so tool outputs remain deterministic

In [9]:
lab.show_stage_pseudocode("Stage 2 - Add calculator", lab.AgentConfig(enable_retrieve=True, enable_tool=True))

### Stage 2 - Add calculator pseudocode

```python
cfg = AgentConfig(
    enable_retrieve=True,
    enable_tool=True,
    enable_evidence_check=False,
)

for item in benchmark:
    evidence = []
    tool_result = None
    answer = None

    while True:
        allowed_tools = ["finish"]
        allowed_tools.append('retrieve')
        allowed_tools.append('grade_calculator')

        if allowed_tools == ["finish"]:
            # Stage 0: there is nothing to choose, so use a direct LLM call.
            answer = llm_text(question_only_prompt)
            break

        decision = planner_llm(item, evidence, tool_result, allowed_tools)

        if decision.action == "retrieve":
            evidence = retrieve(decision.query)
            continue
        if decision.action == "grade_calculator":
            tool_result = grade_calculator(...)
            continue
        if decision.action == "finish":
            answer = render_final_answer(evidence, tool_result)
            break

    if cfg.enable_evidence_check:
        answer = enforce_evidence_gate(answer, evidence, tool_result)
    else:
        # no evidence gate in this stage
        pass

    score_answer(item, answer)
```

In [10]:
stage_2_cfg = lab.AgentConfig(enable_retrieve=True, enable_tool=True)
stage_2_results = lab.run_stage_analysis("Stage 2 - Add calculator", stage_2_cfg)

### Benchmark snapshot

### Benchmark accuracy

Category,Correct,Total,Accuracy
Factual,7,7,100.0%
Cross-doc,2,2,100.0%
Numeric,4,4,100.0%
Abstention,1,3,33.3%
Overall,14,16,87.5%


### Representative correct answer

### Failure cases

### Stage 2 takeaway

The main change here is not just a better score on math-heavy questions.
The workflow changes from "the notebook decides when to compute" to "the model decides whether to retrieve, calculate, or finish."

## 8. Evidence check: what changes in Stage 3

Even with retrieval and tools, the model can still produce unsupported claims.
Stage 3 adds a notebook-side evidence gate after the answer is drafted.

The check is intentionally simple:
- if the answer introduces important question terms that are not supported by the retrieved evidence or tool output, block it
- if the answer is unsupported, replace it with a fixed abstention string

This is a behavior-control layer, not a reasoning-improvement layer.
It mainly helps when the right action is to refuse to guess.

## 9. Stage 3: Add an evidence check

Now we keep the same retrieval/tool loop from Stage 2 and add the evidence gate at the end.

What to watch for:
- unanswerable questions should improve most
- some risky answers should become abstentions
- answerable performance can stay flat or even drop slightly if the gate is too conservative

In [11]:
lab.show_stage_pseudocode("Stage 3 - Add evidence check", lab.AgentConfig(enable_retrieve=True, enable_tool=True, enable_evidence_check=True))

### Stage 3 - Add evidence check pseudocode

```python
cfg = AgentConfig(
    enable_retrieve=True,
    enable_tool=True,
    enable_evidence_check=True,
)

for item in benchmark:
    evidence = []
    tool_result = None
    answer = None

    while True:
        allowed_tools = ["finish"]
        allowed_tools.append('retrieve')
        allowed_tools.append('grade_calculator')

        if allowed_tools == ["finish"]:
            # Stage 0: there is nothing to choose, so use a direct LLM call.
            answer = llm_text(question_only_prompt)
            break

        decision = planner_llm(item, evidence, tool_result, allowed_tools)

        if decision.action == "retrieve":
            evidence = retrieve(decision.query)
            continue
        if decision.action == "grade_calculator":
            tool_result = grade_calculator(...)
            continue
        if decision.action == "finish":
            answer = render_final_answer(evidence, tool_result)
            break

    if cfg.enable_evidence_check:
        answer = enforce_evidence_gate(answer, evidence, tool_result)
    else:
        # no evidence gate in this stage
        pass

    score_answer(item, answer)
```

In [12]:
stage_3_cfg = lab.AgentConfig(enable_retrieve=True, enable_tool=True, enable_evidence_check=True)
stage_3_results = lab.run_stage_analysis("Stage 3 - Add evidence check", stage_3_cfg)

### Benchmark snapshot

### Benchmark accuracy

Category,Correct,Total,Accuracy
Factual,7,7,100.0%
Cross-doc,2,2,100.0%
Numeric,4,4,100.0%
Abstention,3,3,100.0%
Overall,16,16,100.0%


### Representative correct answer

### Failure cases

_No items to display._

### Stage 3 takeaway

This stage is mainly about behavior on unsupported questions.
A system that sometimes refuses to answer can be more useful than one that answers every question with high confidence.

## 10. Cross-stage comparison

This final table separates two different questions:
- how often each stage answers supported questions correctly
- how often each stage refuses unsupported questions appropriately

That distinction matters.
A teaching notebook on agentic systems should not collapse grounded answering and safe abstention into one number without comment.

In [13]:
comparison_df = lab.show_stage_comparison()

### Stage comparison

stage,Answerable overall,Canonical abstention,Cross-doc,Factual,Numeric,Unanswerable
Stage 0 - LLM only,7.7% (1/13),0.0% (0/3),0.0% (0/2),14.3% (1/7),0.0% (0/4),33.3% (1/3)
Stage 1 - Add retrieval,69.2% (9/13),0.0% (0/3),100.0% (2/2),100.0% (7/7),0.0% (0/4),66.7% (2/3)
Stage 2 - Add calculator,100.0% (13/13),0.0% (0/3),100.0% (2/2),100.0% (7/7),100.0% (4/4),66.7% (2/3)
Stage 3 - Add evidence check,100.0% (13/13),100.0% (3/3),100.0% (2/2),100.0% (7/7),100.0% (4/4),100.0% (3/3)
